# ML4VRP 2026 — Colab Bridge
Remote execution environment for the Hierarchical MARL-HGS hybrid solver.

**Runtime:** GPU → T4 (16GB VRAM)

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ML4VRP_2026'
print(f'Project dir: {PROJECT_DIR}')

In [ ]:
# Cell 1.5: Sync project source to Colab
import os, glob, zipfile

src_dir = os.path.join(PROJECT_DIR, 'src')

if os.path.isdir(src_dir):
    py_files = sorted(glob.glob(os.path.join(src_dir, '*.py')))
    print(f'✅ src/ found with {len(py_files)} .py files:')
    for f in py_files:
        print(f'  {os.path.basename(f)}')
else:
    print('src/ not found in PROJECT_DIR.')
    print('Upload a .zip of your project (must contain src/ at top level).')
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(PROJECT_DIR)
    print(f'Extracted {zip_name} to {PROJECT_DIR}')
    assert os.path.isdir(src_dir), f'src/ still not found after extraction in {PROJECT_DIR}'
    py_files = sorted(glob.glob(os.path.join(src_dir, '*.py')))
    print(f'✅ src/ now available with {len(py_files)} .py files.')

In [ ]:
# Cell 2: Install Dependencies
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '')
print(f'PyTorch {TORCH}, CUDA {torch.version.cuda}')

!pip install -q pyvrp gymnasium
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-geometric

In [ ]:
# Cell 3: Verify imports + run smoke tests
import sys, os

LOCAL_DIR = '/content/ml4vrp'
if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)
os.chdir(LOCAL_DIR)

from src.model_vision import GNNEncoder
from src.agent_manager import FleetManager
from src.agent_driver import RouteDriver
print('\u2713 GNNEncoder, FleetManager, RouteDriver imported OK')

!python -m src.main

In [ ]:
# Cell 4: GPU Smoke Test — 400-node FP16
import torch
from src.model_vision import GNNEncoder

device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

encoder = GNNEncoder().to(device)

N = 401  # depot + 400 customers
x = torch.rand(N, 3, device=device)
x[0, 2] = 0.0
pos = x[:, :2]
batch = torch.zeros(N, dtype=torch.long, device=device)

with torch.amp.autocast('cuda'):
    node_emb, graph_emb = encoder(x, pos, batch)

print(f'Node embeddings: {node_emb.shape}, dtype={node_emb.dtype}')
print(f'Graph embedding: {graph_emb.shape}, dtype={graph_emb.dtype}')
print(f'Parameters: {sum(p.numel() for p in encoder.parameters()):,}')

In [ ]:
# Cell 5: Memory Profile — Batch=32 x N=400
import torch
from torch_geometric.data import Data, Batch
from src.model_vision import GNNEncoder

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

encoder = GNNEncoder().to(device)

graphs = []
for _ in range(32):
    n = 401
    coords = torch.rand(n, 2)
    demands = torch.rand(n, 1)
    demands[0] = 0.0
    graphs.append(Data(x=torch.cat([coords, demands], dim=-1), pos=coords))

big_batch = Batch.from_data_list(graphs).to(device)

with torch.amp.autocast('cuda'):
    node_emb, graph_emb = encoder(big_batch.x, big_batch.pos, big_batch.batch)

loss = node_emb.sum() + graph_emb.sum()
loss.backward()

peak_mb = torch.cuda.max_memory_allocated() / 1e6
print(f'Peak GPU memory (batch=32, N=400, FP16): {peak_mb:.1f} MB')
print(f'Fraction of T4 VRAM: {peak_mb / 16384 * 100:.1f}%')

---
## Stage 5: Production Training

In [ ]:
# Cell 6: Download Uchoa X-dataset from ML4VRP2026 GitHub repo
import os, glob, shutil

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Check if already downloaded
existing = glob.glob(os.path.join(DATA_DIR, 'X-n*.vrp'))
if len(existing) >= 50:
    print(f'\u2713 {len(existing)} X-instances already in {DATA_DIR}')
else:
    # Clone the competition repo (sparse checkout for instances only)
    !git clone --depth 1 --filter=blob:none --sparse \
        https://github.com/ML4VRP/ML4VRP2026.git /tmp/ml4vrp_repo
    %cd /tmp/ml4vrp_repo
    !git sparse-checkout set Instances/cvrp/vrp
    %cd /content
    # Copy .vrp files to DATA_DIR
    src = '/tmp/ml4vrp_repo/Instances/cvrp/vrp'
    for f in glob.glob(os.path.join(src, 'X-n*.vrp')):
        shutil.copy(f, DATA_DIR)
    shutil.rmtree('/tmp/ml4vrp_repo', ignore_errors=True)

files = sorted(glob.glob(os.path.join(DATA_DIR, 'X-n*.vrp')))
print(f'\u2713 Ready: {len(files)} instances in {DATA_DIR}')
if files:
    print(f'  Range: {os.path.basename(files[0])} ... {os.path.basename(files[-1])}')

In [ ]:
# Cell 7: TrainingDashboard - Live monitoring
import csv, os, time
import matplotlib.pyplot as plt
from IPython.display import clear_output

class TrainingDashboard:
    def __init__(self, csv_path='logs/training_metrics.csv'):
        self.csv_path = csv_path

    def read_metrics(self):
        data = {}
        if not os.path.exists(self.csv_path):
            return data
        with open(self.csv_path, 'r') as f:
            for row in csv.DictReader(f):
                for k, v in row.items():
                    data.setdefault(k, []).append(float(v))
        return data

    def plot(self):
        data = self.read_metrics()
        if not data or 'epoch' not in data:
            print('No training data yet...'); return
        ep = data['epoch']
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        # NV
        nv = data.get('final_nv', [])
        ax[0].plot(ep, nv, 'b-', lw=1.5)
        ax[0].set_title('Fleet Size (NV)'); ax[0].set_xlabel('Epoch')
        ax[0].grid(True, alpha=0.3)
        if nv: ax[0].axhline(min(nv), color='g', ls='--', alpha=.5, label=f'Best:{min(nv):.0f}'); ax[0].legend()
        # TD
        td = data.get('final_td', [])
        ax[1].plot(ep, td, 'r-', lw=1.5)
        ax[1].set_title('Total Distance (TD)'); ax[1].set_xlabel('Epoch')
        ax[1].grid(True, alpha=0.3)
        if td: ax[1].axhline(min(td), color='g', ls='--', alpha=.5, label=f'Best:{min(td):.0f}'); ax[1].legend()
        # Loss
        ax[2].plot(ep, data.get('mgr_policy_loss',[]), 'b-', lw=1.5, label='Manager')
        ax[2].plot(ep, data.get('drv_policy_loss',[]), 'r-', lw=1.5, label='Driver')
        ax[2].set_title('PPO Policy Loss'); ax[2].set_xlabel('Epoch')
        ax[2].grid(True, alpha=0.3); ax[2].legend()
        plt.tight_layout(); plt.show()
        sc = data.get('final_score',[0])
        print(f'Epochs:{len(ep)} | Best:{min(sc):.0f} | Latest:{sc[-1]:.0f} | NV:{nv[-1]:.0f} | TD:{td[-1]:.0f}')

    def monitor(self, interval=15, max_refreshes=500):
        for _ in range(max_refreshes):
            clear_output(wait=True); self.plot(); time.sleep(interval)

dashboard = TrainingDashboard(os.path.join(LOCAL_DIR, 'logs', 'training_metrics.csv'))
dashboard.plot()

In [ ]:
# Cell 8: Launch Training Run (T4-optimized)
import os, sys
os.chdir(LOCAL_DIR)

!python -m src.main train \
    --instance_path /content/data \
    --epochs 100 \
    --batch_size 16 \
    --manager_lr 1e-4 \
    --driver_lr 5e-4 \
    --episodes_per_epoch 1 \
    --fp16 \
    --ent_coeff 0.05 \
    --log_dir {LOCAL_DIR}/logs \
    --checkpoint_dir {LOCAL_DIR}/checkpoints \
    --save_interval 10 \
    --curriculum_epochs 20

In [ ]:
# Cell 9: Live Dashboard (run after launching training)
dashboard.monitor(interval=15)